In [2]:
import pandas as pd

In [3]:
import requests
import pandas as pd

url = "https://services8.arcgis.com/rGGrs6HCnw87OFOT/arcgis/rest/services/Full_Extract_Environmental_Health_Disparities_Map_Version3/FeatureServer/0/query"

params = {
    "where": "1=1",
    "outFields": "*",
    "f": "json"
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

df = pd.DataFrame(
    feature["attributes"]
    for feature in data["features"]
)

In [4]:
# Load PSRC EFA Definitions
df_efa = pd.read_csv(r"C:\Users\bnichols\Puget Sound Regional Council\GIS - Sharing\Projects\Transportation\RTP_2026\equity_focus_areas\efa_3groupings_1SD\equity_focus_areas_2023.csv")

In [5]:
df = df.merge(df_efa, how='inner', left_on='CensusTract2020', right_on='GEOID20')

In [6]:
# 0: 'Below Regional Average', 
# 1: 'Above Regional Average', 
# 2: 'Higher Share of Equity Population'

In [9]:
import numpy as np
import pandas as pd
from scipy import stats

analysis_cols = ['Pesticide', 'Heavy_Traffic_Roadways', 'Diesel_AirToxics', 'Hazardous_Waste',
                 'Wildfire_Smoke', 'Lead_Risk', 'Superfund_Sites',
                 'Wastewater_Discharge','Water_Quality', 'Food_Environment',
                 'Digital_Infrastructure', 'No_HS_Diploma', 'Transportation_Expense',
                 'Unaffordable_Housing', 'Unemployment', 'Respiratory_Disease',
                #  'CardiovascularDisease_new',
                #  'LowBirthWeight_new'
                 ]
df_analysis = df.loc[:, ['efa_poc'] + analysis_cols].copy()
df_analysis['poc_dummy'] = (df_analysis['efa_poc'] >= 1).astype(int)

rows = []
for col in analysis_cols:
    # print(col)
    poc = df_analysis.loc[df_analysis['poc_dummy'] == 1, col].dropna()
    non = df_analysis.loc[df_analysis['poc_dummy'] == 0, col].dropna()
    ttest_two = stats.ttest_ind(poc, non, equal_var=False, alternative='two-sided')
    mw_two = stats.mannwhitneyu(poc, non, alternative='two-sided')
    rows.append({
        'variable': col,
        'n_poc': len(poc),
        'n_non_poc': len(non),
        'mean_poc': poc.mean(),
        'mean_non_poc': non.mean(),
        'median_poc': poc.median(),
        'median_non_poc': non.median(),
        'mean_diff': poc.mean() - non.mean(),
        'welch_t_p_two_sided': ttest_two.pvalue,
        'mannwhitney_p_two_sided': mw_two.pvalue,
    })

results = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.6g}'.format)
results

,variable,n_poc,n_non_poc,mean_poc,mean_non_poc,median_poc,median_non_poc,mean_diff,welch_t_p_two_sided,mannwhitney_p_two_sided
0,Pesticide,417,502,2.08015,6.50271,0.105001,0.714267,-4.42257,0.0002286,6.66876e-11
1,Heavy_Traffic_Roadways,417,502,100076,56913.4,79887.6,37000,43162.5,1.08911e-22,4.74265e-24
2,Diesel_AirToxics,417,502,94.7441,79.3916,91.3624,74.4812,15.3524,2.0538e-16,1.46283e-20
3,Hazardous_Waste,417,502,6.81612,3.93556,5.50109,2.33859,2.88056,2.89072e-17,3.58277e-24
4,Wildfire_Smoke,417,502,1828.81,1785.39,1803.9,1792.7,43.4212,8.44559e-05,0.0225618
5,Lead_Risk,417,502,25.7865,27.9628,25.7303,27.6286,-2.17633,0.0133239,0.0185588
6,Superfund_Sites,417,502,0.34087,0.173707,0.30367,0.0570989,0.167164,2.66557e-15,4.21044e-18
7,Wastewater_Discharge,417,502,22.9013,2.76855,1.58262,0.165691,20.1327,4.37083e-18,1.46062e-26
8,Water_Quality,417,502,4.15348,4.11952,3,3,0.0339553,0.875832,0.800012
9,Food_Environment,417,502,0.185225,0.200899,0.173709,0.166667,-0.0156739,0.222092,0.568175


In [10]:
results.to_clipboard()